# Conducto 1D estacionario (Calbuco) — **sin IDA**

Misma física que `RIconduitex5_5.py`, pero la marcha en profundidad $z$ usa **RK4 con paso adaptativo** (método de líneas / diferencias finitas en $z$). Solo hace falta **numpy** y **matplotlib** (`requirements-simple.txt`).

Módulo: `RIconduitex5_5_march.py` → función `RIconduitex5_5_march_f`.

### Dependencias

```bash
pip install -r requirements-simple.txt
```

Ejecutar todas las celdas desde la carpeta del repositorio.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from calbuco2015d import radius1, overP1, h2o1, T1, xi1, H, pfinal
from RIconduitex5_5_march import RIconduitex5_5_march_f, graficar_perfil_march

## Caso Calbuco: método de tiro + marcha RK4

Puede tardar ~10–30 s según la máquina (varias iteraciones de tiro).

In [ ]:
radius = radius1
Pressure = overP1

(
    zsol,
    sol,
    count,
    vinicial,
    rho_m,
    rbub,
    visc,
    rho_ti,
    fragcrit,
    phicrit,
    limphi1,
    limphi2,
    co,
    xi,
    xfinal,
    cg,
) = RIconduitex5_5_march_f(radius, Pressure, h2o1, T1, xi1)

In [ ]:
z = np.asarray(zsol, dtype=float)
Y = np.asarray(sol, dtype=float)

print(f"Iteraciones de tiro: {count}")
print(f"Velocidad en la cámara (convergida): {vinicial:.3f} m/s")
print(f"Profundidad boca: {z[-1]:.1f} m  (H cámara = {H:.0f} m)")
print(f"Presión boca: {Y[-1, 0]/1e6:.4f} MPa  (pfinal ≈ {pfinal/1e6:.4f} MPa)")
print(f"φ boca: {Y[-1, 1]:.4f}  |  u_g boca: {Y[-1, 5]:.2f} m/s")

## Perfiles a lo largo del conducto

Hay **dos tramos** concatenados: (1) columna **líquida** φ=0 hasta ~−3,6 km y (2) **bifásico** con marcha RK4. Si se dibuja todo con una sola línea, matplotlib “une” el salto de presión/φ y parece un escalón.

La celda siguiente usa `graficar_perfil_march`: **dos curvas separadas**, φ de **equilibrio con P** en el tramo con gas, y una línea vertical de referencia en la saturación.

In [ ]:
fig, axs, z_plot, Y_plot, ij_bif = graficar_perfil_march(
    zsol,
    sol,
    visc,
    limphi1,
    phicrit,
    co,
    xi,
    rho_m,
)
plt.show()
print(f"Índice inicio bifásico: {ij_bif}  |  z_saturación ≈ {z_plot[ij_bif]:.0f} m")

## (Opcional) Un tramo con `solv`: marcha explícita en $z$

Útil para una tarea: fijar condiciones iniciales y avanzar un tramo sin método de tiro.

In [ ]:
import RIconduitex5_5_march as M
from calbuco2015d import R, beta, C1, g, rcrust, limperl, limperh
from density import density
from calbuco2015d import sio2, tio2, al2o3, feo, mgo, cao, na2o, k2o

M.h2o = h2o1
M.wr = float(radius1)
M.T = float(T1)
M.Tc = T1 - 273.15
M.co = h2o1 / 100.0
M.xi = float(xi1)
M.cg = 8
Pi = rcrust * g * abs(H) + overP1
M.Pi = Pi
dis = min(C1 * Pi ** beta, M.co)
M.rho_m = density(sio2, tio2, al2o3, feo, mgo, cao, na2o, k2o, dis * 100, M.Tc, Pi / 1e6)
M.q = 10.0 * M.rho_m
M.vinicial = 10.0
M.Nd = 1e8
M.xfinal = xi1
M.n_eq = 1
M.phicrit = 0.75
M.limphi1 = limperl
M.limphi2 = limperh
M.xmax = 0.5

y0 = np.array([Pi, 0.05, M.Nd, M.xi, 10.0, 10.0])
Y_seg, Z_seg = M.solv(H, -2500.0, y0, None, 1e-6, 1e-6, 400, 1, cond=None)

fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
ax[0].semilogx(Y_seg[:, 0], Z_seg, "b-")
ax[0].set_xlabel("P (Pa)")
ax[0].set_ylabel("z (m)")
ax[0].set_title("Tramo corto desde la cámara")
ax[0].grid(True, alpha=0.3)
ax[1].plot(Y_seg[:, 1], Z_seg, "r-")
ax[1].set_xlabel("φ")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Puntos en la malla: {len(Z_seg)}  |  z: {Z_seg[0]:.0f} → {Z_seg[-1]:.0f} m")